# PatchTST Baseline — electricity + traffic

Trains **official PatchTST** (d_model=512, e_layers=3, patch_len=16, stride 8, lookback 512, pred_len 48)
on `electricity` (321 channels) and `traffic` (862 channels) for the NanoForecast paper's standard benchmark.

**Window subsampling**: electricity/traffic use stride=1024 (train) / stride=128 (val) — the full protocol
would take days. Documented as a protocol deviation in the paper.

**Runtime**: ~3–5 min total on a T4 GPU.

**Resumable**: If session drops, re-run all cells — picks up from last checkpoint.

**Drive**: checkpoints saved to `MyDrive/nanoforecast-baselines/patchtst/`

## Step 1 — Setup & GPU check

In [ ]:
import torch, sys, os, json, time, numpy as np
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU.'
torch.set_num_threads(1)
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'git+https://github.com/eulogik/NanoForecast.git@v0.5',
                'safetensors', 'pandas'], check=True)

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/nanoforecast-baselines/patchtst'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## Step 2 — Download vendored PatchTST (thuml/Time-Series-Library)

In [ ]:
import urllib.request
TSL = 'https://raw.githubusercontent.com/eulogik/NanoForecast/v0.5/benchmarks/tsl'
TRAIN = 'https://raw.githubusercontent.com/eulogik/NanoForecast/v0.5/benchmarks/train_patchtst.py'
BENCH = 'https://raw.githubusercontent.com/eulogik/NanoForecast/v0.5/benchmark_standard.py'

os.makedirs('benchmarks/tsl', exist_ok=True)
for name in ['PatchTST.py','Embed.py','SelfAttention_Family.py',
             'Transformer_EncDec.py','masking.py','__init__.py']:
    urllib.request.urlretrieve(f'{TSL}/{name}', f'benchmarks/tsl/{name}')
urllib.request.urlretrieve(TRAIN, 'benchmarks/train_patchtst.py')
urllib.request.urlretrieve(BENCH, 'benchmark_standard.py')

sys.path.insert(0, '.')
print('Vendored TSL + train_patchtst.py ready')

## Step 3 — Train electricity

321 channels, stride 1024 (~20 train batches/epoch). ~1 min on T4.

In [ ]:
import torch.nn as nn
from nanoforecast.data.real_datasets import _load_dataframe
from benchmarks.train_patchtst import load_split, windows as make_windows

CTX, H, LR, MAX_EPOCHS, PATIENCE = 512, 48, 1e-4, 100, 3
TRAIN_FRAC = 0.70; TRAIN_VAL_FRAC = 0.80
BATCH = 16; TR_STRIDE = 1024; VA_STRIDE = 128
TOTAL = CTX + H

class Cfg:
    task_name='long_term_forecast'; seq_len=CTX; pred_len=H
    enc_in=1; dec_in=1; c_out=1; d_model=512; n_heads=8
    e_layers=3; d_ff=512; dropout=0.3; factor=3
    activation='gelu'; output_attention=False; embed='timeF'; freq='h'

def make_model(C):
    cfg = Cfg(); cfg.enc_in=C; cfg.dec_in=C; cfg.c_out=C
    from benchmarks.tsl.PatchTST import Model as PM
    return PM(cfg, patch_len=16, stride=8)

def train_one(dataset):
    data = load_split(dataset)
    C = data.x_train.shape[1]
    n_tr, x_tr = make_windows(data.x_train, CTX, H)
    n_va, x_va = make_windows(data.x_val, CTX, H)
    model = make_model(C).to('cuda')
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    out_dir = os.path.join(DRIVE_ROOT, 'checkpoints'); os.makedirs(out_dir, exist_ok=True)
    resume_path = os.path.join(out_dir, f'{dataset}_resume.pt')
    final_path  = os.path.join(out_dir, f'{dataset}.pt')
    meta_path   = os.path.join(out_dir, f'{dataset}.json')
    if os.path.exists(final_path) and not os.path.exists(resume_path):
        print(f'[{dataset}] checkpoint exists, skipping'); return None
    start_epoch, best_va, best_state, bad = 0, float('inf'), None, 0
    if os.path.exists(resume_path):
        r = torch.load(resume_path, map_location='cuda')
        model.load_state_dict(r['model']); opt.load_state_dict(r['opt'])
        np.random.set_state(r['rng']); start_epoch, best_va, bad = r['epoch'], r['best_va'], r['bad']
        best_state = r.get('best_state'); print(f'[{dataset}] resumed epoch {start_epoch}')
    def save_resume(ep):
        torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'rng':np.random.get_state(),
                    'epoch':ep+1,'best_va':best_va,'bad':bad,'best_state':best_state}, resume_path)
    def gather(x, sel):
        w = x[sel[:, None] + np.arange(TOTAL)]
        return (torch.from_numpy(np.ascontiguousarray(w[:,:CTX,:])).to('cuda'),
                torch.from_numpy(np.ascontiguousarray(w[:,CTX:,:])).to('cuda'))
    def evaluate(starts):
        model.eval(); tot,cnt=0.0,0
        with torch.no_grad():
            for i in range(0, starts.shape[0], BATCH):
                sel=starts[i:i+BATCH]; b,tgt=gather(x_va,sel)
                out=model(b,None,None,None); tot+=loss_fn(out,tgt).item()*out.shape[0]; cnt+=out.shape[0]
        return tot/max(cnt,1)
    va_starts = np.arange(0, n_va, VA_STRIDE)
    t0=time.time()
    for epoch in range(start_epoch, MAX_EPOCHS):
        torch.cuda.empty_cache(); model.train()
        perm=np.random.permutation(n_tr)[::TR_STRIDE]; tot,cnt=0.0,0
        for i in range(0, perm.shape[0], BATCH):
            sel=perm[i:i+BATCH]; b,tgt=gather(x_tr,sel)
            out=model(b,None,None,None); loss=loss_fn(out,tgt)
            opt.zero_grad(); loss.backward(); opt.step()
            tot+=loss.item()*sel.shape[0]; cnt+=sel.shape[0]
        tr_l=tot/cnt; va_l=evaluate(va_starts)
        msg=f'  epoch {epoch:3d} | train {tr_l:.6f} | val {va_l:.6f}'
        if va_l<best_va: best_va=va_l; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0; msg+=' * BEST'
        else: bad+=1
        print(f'[{dataset}] {msg}  ({time.time()-t0:.0f}s)'); save_resume(epoch)
        if bad>=PATIENCE: break
    if os.path.exists(resume_path): os.remove(resume_path)
    torch.save(best_state, final_path)
    meta={'dataset':dataset,'n_vars':C,'means':data.means.tolist(),'stds':data.stds.tolist(),
          'best_val_mse':best_va,'epochs':epoch+1,
          'window_stride':{'train':TR_STRIDE,'val':VA_STRIDE},
          'arch':{'d_model':512,'d_ff':512,'n_heads':8,'e_layers':3,'patch_len':16,
                  'stride':8,'dropout':0.3,'seq_len':CTX,'pred_len':H},
          'train_seconds':round(time.time()-t0,1)}
    with open(meta_path,'w') as fh: json.dump(meta,fh,indent=1)
    print(f'[{dataset}] DONE best val {best_va:.6f} in {epoch+1} eps ({meta["train_seconds"]}s)')
    return meta

# Run electricity
BATCH = 8  # 321 channels × 512 dim × 512 FFN → needs small batch on T4
r = train_one('electricity')
if r: print(json.dumps(r, indent=2))

## Step 4 — Train traffic

862 channels, stride 1024 (~12 train batches/epoch). ~2 min on T4.

In [ ]:
BATCH = 4  # 862 channels × 512 dim → needs very small batch on T4
r = train_one('traffic')
if r: print(json.dumps(r, indent=2))

print('\n=== ALL DONE ===')
print('Checkpoints:', os.listdir(os.path.join(DRIVE_ROOT, 'checkpoints')))

## Step 5 — Verify saved checkpoints

Check that `electricity.pt`, `electricity.json`, `traffic.pt`, `traffic.json` exist in Drive.
Copy them to your local machine or run `benchmark_standard.py --models patchtst` locally.

In [ ]:
ck_dir = os.path.join(DRIVE_ROOT, 'checkpoints')
for f in sorted(os.listdir(ck_dir)):
    sz = os.path.getsize(os.path.join(ck_dir, f))
    print(f'  {f:30s} {sz/1024:.1f} KB')

# Print summary
for ds in ['electricity', 'traffic']:
    meta_path = os.path.join(ck_dir, f'{ds}.json')
    if os.path.exists(meta_path):
        m = json.load(open(meta_path))
        print(f'\n{ds}: {m["n_vars"]} vars, {m["epochs"]} epochs, '
              f'best_val_mse={m["best_val_mse"]:.6f}, {m["train_seconds"]}s')
    else:
        print(f'\n{ds}: NOT FOUND')